# SIMBAC-NFS — Interactive Demo

This notebook demonstrates **SIMBAC-NFS** end-to-end on the **Yacht Hydrodynamics** dataset (UCI ID 243).
No local data files are needed — the dataset is fetched live from the UCI repository.

**What you will see:**
1. Load data from UCI
2. Build the hybrid pool (T boosting rounds × M bagging per round)
3. Compress with SIMBA into a single compact fuzzy model
4. Evaluate: RMSE, MAE, R²
5. Print the fuzzy rule base as IF-THEN statements
6. Visualize membership functions per feature
7. Interpretability summary

## 1. Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

# Add repo root to path so src/ imports work
sys.path.insert(0, ".")

from src.models.fcm_tsk import FCMTSKModel
from src.models.bagging_ensemble import BaggingFCMTSK
from src.models.compression import GradNFSCompressor

print("Imports OK")

## 2. Load Data from UCI

**Yacht Hydrodynamics** (n=308, p=6) — predicts residuary resistance of sailing yachts  
from hull geometry and operational parameters.

In [ ]:
repo = fetch_ucirepo(id=243)          # Yacht Hydrodynamics
X_df = repo.data.features
y_ser = repo.data.targets

feature_names = X_df.columns.tolist()
X = X_df.values.astype(float)
y = y_ser.values.flatten().astype(float)

print(f"Dataset : Yacht Hydrodynamics")
print(f"Samples : {len(X)}")
print(f"Features: {feature_names}")
print(f"Target  : residuary_resistance  [{y.min():.2f}, {y.max():.2f}]")

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)} samples   Test: {len(X_test)} samples")

## 4. Phase 1 — Build Hybrid Pool

We run **T=5 boosting rounds**, each training **M=3 bootstrap-resampled FCM-TSK models**.  
The pool has T×M = 15 base learners total.  
Each round is trained on the **residuals** left by the previous rounds.

In [ ]:
T   = 5      # boosting rounds
M   = 3      # bags per round
LR  = 0.3    # boosting learning rate
NC  = 15     # FCM rule budget per base learner

y_res = y_train.copy().astype(float)
pool  = []

for t in range(T):
    bag = BaggingFCMTSK(
        n_estimators=M,
        n_rules=NC,
        min_rules=3,
        base_params={"mf_type": "gaussian"},
        random_state=42 + t,
    )
    bag.fit(X_train, y_res)

    round_preds = np.mean(
        [e.predict(X_train) for e in bag.estimators_], axis=0
    )
    y_res = y_res - LR * round_preds
    pool.extend(bag.estimators_)

    total_rules = sum(e.n_rules for e in bag.estimators_)
    print(f"  Round {t+1}: {len(bag.estimators_)} learners, "
          f"{total_rules} rules, "
          f"residual RMSE = {np.sqrt(np.mean(y_res**2)):.4f}")

print(f"\nPool: {len(pool)} base learners, "
      f"{sum(m.n_rules for m in pool)} total rules")

## 5. Phase 2 — SIMBA Compression

All pool rules are projected into a common space, weighted by model performance and rule support,  
and clustered by antecedent similarity.  Redundant clusters are merged; consequents are re-fitted  
by Ridge regression on the consolidated rule structure.

In [ ]:
compressor = GradNFSCompressor(
    tau=0.95,                       # similarity threshold for clustering
    similarity_method="combined",   # Bhattacharyya + Wasserstein + centroid
    weight_by_performance=True,
    cumulative_importance=0.99,     # keep rules covering 99% of importance
    min_rules=3,
    refit_consequents=True,
    tune_refit_alpha=True,
)

model = compressor.compress(pool, X_train, y_train)

meta = model.compression_meta_
print(f"Pool rules   : {meta['n_rules_ensemble_total']}")
print(f"Compressed to: {meta['n_rules_after']} rules")
print(f"Compression  : {meta['compression_pct']}")

## 6. Prediction Performance

In [ ]:
y_pred = model.predict(X_test)

rmse = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))
mae  = float(np.mean(np.abs(y_test - y_pred)))
r2   = float(1 - np.sum((y_test - y_pred) ** 2)
             / (np.sum((y_test - np.mean(y_test)) ** 2) + 1e-12))

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

## 7. Fuzzy Rule Base — IF-THEN Printout

Each rule maps linguistic labels (LOW / MEDIUM-LOW / MEDIUM / MEDIUM-HIGH / HIGH)  
to normalized feature positions. Labels are derived from the centroid location in [0, 1].

In [ ]:
labels = model.get_linguistic_labels(feature_names)

print(f"=== Fuzzy Rule Base ({model.n_rules} rules) ===\n")
for i, rule in enumerate(labels):
    antecedent = " AND ".join(
        f"{feat} is {lbl}" for feat, lbl in rule.items()
    )
    print(f"Rule {i+1:2d}: IF {antecedent}")

## 8. Membership Function Visualization

Each Gaussian membership function defines how strongly an input value fires a rule.  
Overlapping curves mean those rules respond to similar feature values.  
Fewer, well-separated curves = a more interpretable partition.

In [ ]:
n_feat = len(feature_names)
cols   = min(n_feat, 3)
rows   = (n_feat + cols - 1) // cols

fig, axes = plt.subplots(rows, cols,
                          figsize=(4.5 * cols, 3.0 * rows),
                          squeeze=False)
x_vals = np.linspace(0, 1, 300)

# Color palette — one color per rule
colors = plt.cm.tab10(np.linspace(0, 1, model.n_rules))

for j, fname in enumerate(feature_names):
    ax = axes[j // cols][j % cols]
    for r in range(model.n_rules):
        c = float(model.centers_[r, j])
        s = float(max(model.sigmas_[r, j], 1e-3))
        mf = np.exp(-0.5 * ((x_vals - c) / s) ** 2)
        ax.plot(x_vals, mf, color=colors[r], alpha=0.75, linewidth=1.5)
        ax.axvline(c, color=colors[r], alpha=0.25, linestyle="--", linewidth=0.8)
    ax.set_title(fname, fontsize=9, fontweight="bold")
    ax.set_xlabel("Normalized value", fontsize=8)
    ax.set_ylabel("Firing strength", fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.08)
    ax.tick_params(labelsize=7)

# Hide unused axes
for j in range(n_feat, rows * cols):
    axes[j // cols][j % cols].set_visible(False)

plt.suptitle(
    f"SIMBAC-NFS Membership Functions — {model.n_rules} rules "
    f"(Yacht Hydrodynamics)",
    fontsize=11, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

## 9. Prediction vs Ground Truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Scatter: predicted vs actual
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidths=0.4, s=40)
lim = [min(y_test.min(), y_pred.min()) - 1,
       max(y_test.max(), y_pred.max()) + 1]
ax.plot(lim, lim, "r--", linewidth=1.2, label="Perfect fit")
ax.set_xlabel("Actual residuary resistance", fontsize=10)
ax.set_ylabel("Predicted", fontsize=10)
ax.set_title(f"Predicted vs Actual  (R² = {r2:.4f})", fontsize=10)
ax.legend(fontsize=9)

# Residual plot
ax = axes[1]
residuals = y_test - y_pred
ax.scatter(y_pred, residuals, alpha=0.6, edgecolors="k", linewidths=0.4, s=40)
ax.axhline(0, color="r", linestyle="--", linewidth=1.2)
ax.set_xlabel("Predicted", fontsize=10)
ax.set_ylabel("Residual", fontsize=10)
ax.set_title("Residuals", fontsize=10)

plt.suptitle("SIMBAC-NFS — Yacht Hydrodynamics", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Interpretability Summary

In [ ]:
pool_rules  = meta["n_rules_ensemble_total"]
final_rules = meta["n_rules_after"]
compression = 100.0 * (1 - final_rules / pool_rules)
stability   = float(np.mean(meta["rule_stability_scores"]))

print("=" * 42)
print("  SIMBAC-NFS Interpretability Summary")
print("=" * 42)
print(f"  Pool size (T×M base learners)  : {len(pool):>6}")
print(f"  Total pool rules               : {pool_rules:>6}")
print(f"  Compressed rules               : {final_rules:>6}")
print(f"  Rule reduction                 : {compression:>5.1f}%")
print(f"  Mean rule stability            : {stability:>6.4f}")
print(f"  Similarity threshold (τ)       : {meta['tau']:>6.2f}")
print(f"  Refit Ridge α                  : {meta.get('refit_alpha', 'N/A')}")
print("-" * 42)
print(f"  Test RMSE                      : {rmse:>8.4f}")
print(f"  Test MAE                       : {mae:>8.4f}")
print(f"  Test R²                        : {r2:>8.4f}")
print("=" * 42)